In [ ]:
from dotenv import load_dotenv
from tqdm.notebook import tqdm
from pathlib import Path
import pandas as pd
from src.utils import split_dataframe, evaluate_baseline, find_optimal_thresholds
import os
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, PredefinedSplit
from sklearn.feature_selection import SelectPercentile, chi2
import joblib
from scipy.sparse import vstack
import json
import numpy as np

tqdm.pandas()
load_dotenv()
SEED = int(os.getenv("SEED", "42"))

## TF-IDF alapú Logistic Regression tanítása gyakori ICD-10-CM chapter osztályozásra (Training a TF-IDF-based Logistic Regression for frequent ICD-10-CM chapter classification)

In [ ]:
THRESHOLD = 0.5
THRESHOLD_TUNE = True
WEIGHT = True

In [ ]:
processed_data_dir = Path("../data/processed")
extension = Path(".parquet")

train_data_dirs = [
    processed_data_dir / "frequent_chapter/without_dropped_sections",
    processed_data_dir / "frequent_chapter/with_dropped_sections",
]

for dir_path in train_data_dirs:
    print(f"\nDirectory: {dir_path.as_posix()}")
    
    if dir_path.exists() and dir_path.is_dir():
        parquet_files = list(dir_path.glob(f"*{extension}"))        
        if parquet_files:
            for file in parquet_files:
                print(f"{file.as_posix()}")
        else:
            print("No .parquet files found in this directory.")
    else:
        print(f"Directory not found: {dir_path}")

In [ ]:
dataset_path = Path("[PATH]")

file_stem = Path(dataset_path).stem
sub_folders = Path(dataset_path).parent.relative_to(processed_data_dir)

if WEIGHT:
    file_stem += "_weighted"
if THRESHOLD_TUNE:
    file_stem += "_threshold_tuned"

baseline_model_path = Path("../models/baseline") / sub_folders / file_stem

print(f"Dataset path: {dataset_path.as_posix()}")
print(f"Model save path: {baseline_model_path.as_posix()}")

### Adathalmaz betöltése és páciens-szintű, stratifikált felosztása tanító-, validációs- és teszthalmazra (Dataset loading and patient-level stratified split into train, validation, and test sets)

In [ ]:
df = pd.read_parquet(dataset_path, engine='pyarrow')
df_train, df_val, df_test, mlb, _  = split_dataframe(df, "chapter", "subject_id", 10, SEED)

In [ ]:
X_train_text = df_train["text"]
y_train = mlb.transform(df_train["chapter"])

X_val_text = df_val["text"]
y_val = mlb.transform(df_val["chapter"])

X_test_text = df_test["text"]
y_test = mlb.transform(df_test["chapter"])

In [ ]:
split_index = [-1] * len(X_train_text) + [0] * len(X_val_text)
pds = PredefinedSplit(test_fold=split_index)

X_combined = pd.concat([X_train_text, X_val_text], axis=0)
y_combined = vstack([y_train, y_val])

In [ ]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('selector', SelectPercentile(chi2)),
    ('clf', OneVsRestClassifier(LogisticRegression()))
])

In [ ]:
parameters = {
    'tfidf__sublinear_tf': [True, False],
    'tfidf__max_features': [2500, 5000, 7500, 10000, 12500],
    'tfidf__ngram_range': [(1, 1), (1,2), (1,3), (2,2), (2,3)],
    'tfidf__max_df': [0.80, 0.85, 0.90, 0.95, 1.0],
    'tfidf__min_df': [0.0001, 0.001, 0.01],
    
    'selector__percentile': [80, 85, 90, 100],
    
    'clf__estimator__solver': ['liblinear'],
    'clf__estimator__max_iter': [5000],
    'clf__estimator__random_state': [SEED],
    'clf__estimator__class_weight': ["balanced"] if WEIGHT else [None],
    'clf__estimator__C': [0.1, 1, 10],
}

In [ ]:
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=parameters,
    cv=pds,  
    scoring='f1_micro',
    refit=False,
    n_jobs=6,
)

In [ ]:
grid_search.fit(X_combined, y_combined)

In [ ]:
print("\nBest parameters:")
print(grid_search.best_params_)

print("\nBest score:")
print(grid_search.best_score_)

In [ ]:
final_model = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('selector', SelectPercentile(chi2)),
    ('clf', OneVsRestClassifier(LogisticRegression()))
])

final_model.set_params(**grid_search.best_params_)

In [ ]:
final_model.fit(X_train_text, y_train)

In [ ]:
best_model_path = baseline_model_path / "best_model"

In [ ]:
best_model_path.mkdir(parents=True, exist_ok=True)

joblib.dump(final_model, best_model_path / "pipeline.joblib")

joblib.dump(mlb, best_model_path / "mlb.joblib")

In [ ]:
eval_dir = baseline_model_path / "val_results"
eval_dir.mkdir(parents=True, exist_ok=True)

metrics, report, y_true, y_probs = evaluate_baseline(final_model, X_val_text, y_val, mlb, THRESHOLD)
metrics.to_json(eval_dir / "metrics.json", orient='records', indent=1)
report.to_json(eval_dir / "classification_report.json", indent=1)

In [ ]:
if THRESHOLD_TUNE:
    thresholds_per_label = find_optimal_thresholds(y_true, y_probs, mlb)
    
else:
    thresholds_per_label = np.full(len(mlb.classes_), THRESHOLD)
    
with open(best_model_path / "thresholds.json", "w") as f: 
    json.dump(thresholds_per_label.tolist(), f) 

In [ ]:
eval_dir = baseline_model_path / "test_results"
eval_dir.mkdir(parents=True, exist_ok=True)

metrics, report, _, _ = evaluate_baseline(final_model, X_test_text, y_test, mlb, thresholds_per_label)
metrics.to_json(eval_dir / "metrics.json", orient='records', indent=1)
report.to_json(eval_dir / "classification_report.json", indent=1)

In [ ]:
with open(best_model_path / "best_params.json", "w") as f:
    json.dump(grid_search.best_params_, f, indent=4)

In [ ]:
vectorizer = final_model.named_steps['tfidf']
selector = final_model.named_steps['selector']

X_train_tfidf = vectorizer.transform(X_train_text)
X_train_selected = selector.transform(X_train_tfidf)

expected_values = np.asarray(X_train_selected.mean(axis=0)).flatten()

joblib.dump(expected_values, best_model_path / "expected_values.joblib")

## TF-IDF alapú Logistic Regression tanítása top 50 ICD-10-CM kód osztályozásra (Training a TF-IDF-based Logistic Regression for top 50 ICD-10-CM code classification)

In [ ]:
THRESHOLD = 0.5
THRESHOLD_TUNE = True
WEIGHT = True

In [ ]:
processed_data_dir = Path("../data/processed")
extension = Path(".parquet")

train_data_dirs = [
    processed_data_dir / "top_50_code/without_dropped_sections",
    processed_data_dir / "top_50_code/with_dropped_sections",
]

for dir_path in train_data_dirs:
    print(f"\nDirectory: {dir_path.as_posix()}")

    if dir_path.exists() and dir_path.is_dir():
        parquet_files = list(dir_path.glob(f"*{extension}"))
        if parquet_files:
            for file in parquet_files:
                print(f"{file.as_posix()}")
        else:
            print("No .parquet files found in this directory.")
    else:
        print(f"Directory not found: {dir_path}")

In [ ]:
dataset_path = Path("[PATH]")

file_stem = Path(dataset_path).stem
sub_folders = Path(dataset_path).parent.relative_to(processed_data_dir)

if WEIGHT:
    file_stem += "_weighted"
if THRESHOLD_TUNE:
    file_stem += "_threshold_tuned"

baseline_model_path = Path("../models/baseline") / sub_folders / file_stem

print(f"Dataset path: {dataset_path.as_posix()}")
print(f"Model save path: {baseline_model_path.as_posix()}")

### Adathalmaz betöltése és páciens-szintű, stratifikált felosztása tanító-, validációs- és teszthalmazra (Dataset loading and patient-level stratified split into train, validation, and test sets)

In [ ]:
df = pd.read_parquet(dataset_path, engine='pyarrow')
df_train, df_val, df_test, mlb, _ = split_dataframe(df, "icd_code", "subject_id", 10, SEED)

In [ ]:
X_train_text = df_train["text"]
y_train = mlb.transform(df_train["icd_code"])

X_val_text = df_val["text"]
y_val = mlb.transform(df_val["icd_code"])

X_test_text = df_test["text"]
y_test = mlb.transform(df_test["icd_code"])

In [ ]:
split_index = [-1] * len(X_train_text) + [0] * len(X_val_text)
pds = PredefinedSplit(test_fold=split_index)

X_combined = pd.concat([X_train_text, X_val_text], axis=0)
y_combined = vstack([y_train, y_val])

In [ ]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('selector', SelectPercentile(chi2)),
    ('clf', OneVsRestClassifier(LogisticRegression()))
])

In [ ]:
parameters = {
    'tfidf__sublinear_tf': [True, False],
    'tfidf__max_features': [2500, 5000, 7500, 10000, 12500],
    'tfidf__ngram_range': [(1, 1), (1,2), (1,3), (2,2), (2,3)],
    'tfidf__max_df': [0.80, 0.85, 0.90, 0.95, 1.0],
    'tfidf__min_df': [0.0001, 0.001, 0.01],
    
    'selector__percentile': [80, 85, 90, 100],
    
    'clf__estimator__solver': ['liblinear'],
    'clf__estimator__max_iter': [5000],
    'clf__estimator__random_state': [SEED],
    'clf__estimator__class_weight': ["balanced"] if WEIGHT else [None],
    'clf__estimator__C': [0.1, 1, 10],
}

In [ ]:
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=parameters,
    cv=pds,  
    scoring='f1_micro',
    refit=False,
    n_jobs=6,
)

In [ ]:
grid_search.fit(X_combined, y_combined)

In [ ]:
print("\nBest parameters:")
print(grid_search.best_params_)

print("\nBest score:")
print(grid_search.best_score_)

In [ ]:
final_model = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('selector', SelectPercentile(chi2)),
    ('clf', OneVsRestClassifier(LogisticRegression()))
])

final_model.set_params(**grid_search.best_params_)

In [ ]:
final_model.fit(X_train_text, y_train)

In [ ]:
best_model_path = baseline_model_path / "best_model"

In [ ]:
best_model_path.mkdir(parents=True, exist_ok=True)

joblib.dump(final_model, best_model_path / "pipeline.joblib")

joblib.dump(mlb, best_model_path / "mlb.joblib")

In [ ]:
eval_dir = baseline_model_path / "val_results"
eval_dir.mkdir(parents=True, exist_ok=True)

metrics, report, y_true, y_probs = evaluate_baseline(final_model, X_val_text, y_val, mlb, THRESHOLD)
metrics.to_json(eval_dir / "metrics.json", orient='records', indent=1)
report.to_json(eval_dir / "classification_report.json", indent=1)

In [ ]:
if THRESHOLD_TUNE:
    thresholds_per_label = find_optimal_thresholds(y_true, y_probs, mlb)

else:
    thresholds_per_label = np.full(len(mlb.classes_), THRESHOLD)

with open(best_model_path / "thresholds.json", "w") as f:
    json.dump(thresholds_per_label.tolist(), f)

In [ ]:
eval_dir = baseline_model_path / "test_results"
eval_dir.mkdir(parents=True, exist_ok=True)

metrics, report, _, _ = evaluate_baseline(final_model, X_test_text, y_test, mlb, thresholds_per_label)
metrics.to_json(eval_dir / "metrics.json", orient='records', indent=1)
report.to_json(eval_dir / "classification_report.json", indent=1)

In [ ]:
with open(best_model_path / "best_params.json", "w") as f:
    json.dump(grid_search.best_params_, f, indent=4)

In [ ]:
vectorizer = final_model.named_steps['tfidf']
selector = final_model.named_steps['selector']

X_train_tfidf = vectorizer.transform(X_train_text)
X_train_selected = selector.transform(X_train_tfidf)

expected_values = np.asarray(X_train_selected.mean(axis=0)).flatten()

joblib.dump(expected_values, best_model_path / "expected_values.joblib")